In [ ]:

# Colab installation cell
!pip -q install rdkit-pypi gensim==4.3.3 scikit-learn pandas numpy scipy matplotlib joblib openpyxl


In [ ]:

import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import dump

from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Crippen, Lipinski, QED, rdMolDescriptors
from gensim.models import Word2Vec

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, matthews_corrcoef,
    confusion_matrix, roc_curve, precision_recall_curve,
    brier_score_loss
)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

OUTPUT_DIR = Path("mol2vec_svm_hiv7_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Output directory:", OUTPUT_DIR.resolve())


## 1. Configuration

In [ ]:

CONFIG = {
    "dataset_url": "https://raw.githubusercontent.com/McahitKutsal/hivcsv/main/HIV7.csv",
    "split": {
        "train": 0.72,
        "validation": 0.10,
        "test": 0.18
    },
    "cv_folds": 5,
    "mol2vec": {
        "radius": 1,
        "vector_size": 300,
        "window": 10,
        "min_count": 1,
        "sg": 1,              # skip-gram
        "negative": 5,
        "epochs": 30,
        "workers": 1          # deterministic/reproducible
    },
    "svm": {
        "kernel": "rbf",
        "C": 1.0,
        "gamma": "scale",
        "probability": True,
        "class_weight": "balanced",
        "random_state": SEED
    },
    "threshold_selection_metric": "f1"
}

print(json.dumps(CONFIG, indent=2))


## 2. Load and clean the HIV7 dataset

In [ ]:

df_raw = pd.read_csv(CONFIG["dataset_url"])
print("Raw shape:", df_raw.shape)
print("Columns:", list(df_raw.columns))
display(df_raw.head())


In [ ]:

def detect_columns(df):
    lower_map = {c.lower().strip(): c for c in df.columns}

    smiles_candidates = ["smiles", "canonical_smiles", "mol", "molecule"]
    label_candidates = ["hiv_active", "label", "activity", "active", "y", "target", "class"]

    smiles_col = next((lower_map[c] for c in smiles_candidates if c in lower_map), None)
    label_col = next((lower_map[c] for c in label_candidates if c in lower_map), None)

    if smiles_col is None:
        for c in df.columns:
            sample = df[c].dropna().astype(str).head(30)
            valid = sum(Chem.MolFromSmiles(x) is not None for x in sample)
            if len(sample) and valid / len(sample) >= 0.7:
                smiles_col = c
                break

    if label_col is None:
        for c in df.columns:
            vals = set(pd.to_numeric(df[c], errors="coerce").dropna().unique())
            if vals and vals.issubset({0, 1}):
                label_col = c
                break

    if smiles_col is None or label_col is None:
        raise ValueError(
            f"Could not detect SMILES/label columns. Available columns: {list(df.columns)}"
        )
    return smiles_col, label_col


def canonicalize_smiles(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, canonical=True)


SMILES_COL, LABEL_COL = detect_columns(df_raw)
print("Detected SMILES column:", SMILES_COL)
print("Detected label column:", LABEL_COL)

df = df_raw[[SMILES_COL, LABEL_COL]].copy()
df.columns = ["smiles_original", "label"]
df["label"] = pd.to_numeric(df["label"], errors="coerce")
df["smiles"] = df["smiles_original"].map(canonicalize_smiles)

df = (
    df.dropna(subset=["smiles", "label"])
      .query("label in [0, 1]")
      .drop_duplicates(subset="smiles", keep="first")
      .reset_index(drop=True)
)
df["label"] = df["label"].astype(int)
df["compound_id"] = [f"HIV7_{i:05d}" for i in range(len(df))]

print("Cleaned shape:", df.shape)
print(df["label"].value_counts().sort_index())
display(df.head())


## 3. Fixed 72/10/18 stratified partition

In [ ]:

# First isolate the independent 18% test set.
development_df, test_df = train_test_split(
    df,
    test_size=CONFIG["split"]["test"],
    stratify=df["label"],
    random_state=SEED
)

# From the remaining 82%, allocate 10% of the full data to validation.
validation_fraction_of_development = (
    CONFIG["split"]["validation"] /
    (CONFIG["split"]["train"] + CONFIG["split"]["validation"])
)

train_df, validation_df = train_test_split(
    development_df,
    test_size=validation_fraction_of_development,
    stratify=development_df["label"],
    random_state=SEED
)

train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

split_summary = pd.DataFrame({
    "subset": ["Training", "Validation", "Test", "Total"],
    "n": [len(train_df), len(validation_df), len(test_df), len(df)],
    "active": [
        int(train_df.label.sum()),
        int(validation_df.label.sum()),
        int(test_df.label.sum()),
        int(df.label.sum())
    ]
})
split_summary["inactive"] = split_summary["n"] - split_summary["active"]
split_summary["fraction"] = split_summary["n"] / len(df)

display(split_summary)
split_summary.to_csv(OUTPUT_DIR / "data_split_summary.csv", index=False)



## 4. True Mol2Vec feature generation

The molecular “sentence” is generated from the ordered Morgan substructure identifiers at radii 0 through \(r\). Each identifier is treated as a token. A skip-gram Word2Vec model learns dense token embeddings, and a molecular vector is obtained by summing the vectors of its recognized substructure tokens.

This is distinct from supplying a 2048-bit Morgan fingerprint directly to the SVM.


In [ ]:

def mol2vec_sentence(smiles, radius=1):
    """Return ordered Morgan identifier tokens for one molecule."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return []

    bit_info = {}
    AllChem.GetMorganFingerprint(mol, radius, bitInfo=bit_info)

    atom_radius_to_identifier = {}
    for identifier, occurrences in bit_info.items():
        for atom_index, rad in occurrences:
            atom_radius_to_identifier[(atom_index, rad)] = str(identifier)

    sentence = []
    for atom_index in range(mol.GetNumAtoms()):
        for rad in range(radius + 1):
            token = atom_radius_to_identifier.get((atom_index, rad))
            if token is not None:
                sentence.append(token)
    return sentence


def train_mol2vec(smiles_series, config, seed=SEED):
    sentences = [
        mol2vec_sentence(s, radius=config["radius"])
        for s in smiles_series
    ]

    if not any(sentences):
        raise ValueError("No valid Mol2Vec sentences were generated.")

    model = Word2Vec(
        sentences=sentences,
        vector_size=config["vector_size"],
        window=config["window"],
        min_count=config["min_count"],
        sg=config["sg"],
        negative=config["negative"],
        workers=config["workers"],
        seed=seed,
        epochs=config["epochs"]
    )
    return model


def molecule_vector(smiles, model, radius=1):
    tokens = mol2vec_sentence(smiles, radius=radius)
    known_vectors = [model.wv[token] for token in tokens if token in model.wv]

    if not known_vectors:
        return np.zeros(model.vector_size, dtype=np.float32)

    # Original Mol2Vec-style molecular representation: sum token vectors.
    return np.sum(known_vectors, axis=0).astype(np.float32)


def featurize_smiles(smiles_series, model, radius=1):
    return np.vstack([
        molecule_vector(s, model=model, radius=radius)
        for s in smiles_series
    ])


# Sanity check
example_sentence = mol2vec_sentence(train_df.loc[0, "smiles"], radius=CONFIG["mol2vec"]["radius"])
print("Number of Mol2Vec tokens in example molecule:", len(example_sentence))
print("First tokens:", example_sentence[:10])


## 5. five-fold cross-validation 

In [ ]:

def calculate_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (np.asarray(y_prob) >= threshold).astype(int)
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_true, y_prob),
        "PR_AUC": average_precision_score(y_true, y_prob),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "Brier": brier_score_loss(y_true, y_prob),
        "TN": confusion_matrix(y_true, y_pred, labels=[0, 1])[0, 0],
        "FP": confusion_matrix(y_true, y_pred, labels=[0, 1])[0, 1],
        "FN": confusion_matrix(y_true, y_pred, labels=[0, 1])[1, 0],
        "TP": confusion_matrix(y_true, y_pred, labels=[0, 1])[1, 1]
    }


skf = StratifiedKFold(
    n_splits=CONFIG["cv_folds"],
    shuffle=True,
    random_state=SEED
)

fold_results = []
oof_prob = np.full(len(train_df), np.nan)
oof_fold = np.full(len(train_df), -1)

for fold, (fit_idx, fold_val_idx) in enumerate(
    skf.split(train_df["smiles"], train_df["label"]), start=1
):
    fold_train = train_df.iloc[fit_idx]
    fold_val = train_df.iloc[fold_val_idx]

    print(f"Fold {fold}: train={len(fold_train)}, held-out={len(fold_val)}")

    # Representation learning only from the current fold-training molecules.
    fold_m2v = train_mol2vec(
        fold_train["smiles"],
        config=CONFIG["mol2vec"],
        seed=SEED + fold
    )

    X_fit = featurize_smiles(
        fold_train["smiles"],
        fold_m2v,
        radius=CONFIG["mol2vec"]["radius"]
    )
    X_fold_val = featurize_smiles(
        fold_val["smiles"],
        fold_m2v,
        radius=CONFIG["mol2vec"]["radius"]
    )

    svm_pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(**CONFIG["svm"]))
    ])

    svm_pipeline.fit(X_fit, fold_train["label"])
    fold_prob = svm_pipeline.predict_proba(X_fold_val)[:, 1]

    oof_prob[fold_val_idx] = fold_prob
    oof_fold[fold_val_idx] = fold

    metrics = calculate_metrics(
        fold_val["label"].to_numpy(),
        fold_prob,
        threshold=0.5
    )
    metrics["Fold"] = fold
    metrics["n_train"] = len(fold_train)
    metrics["n_validation"] = len(fold_val)
    metrics["vocabulary_size"] = len(fold_m2v.wv)
    fold_results.append(metrics)

fold_results_df = pd.DataFrame(fold_results)
display(fold_results_df)
fold_results_df.to_csv(OUTPUT_DIR / "mol2vec_svm_5fold_metrics.csv", index=False)


In [ ]:

metric_columns = [
    "Accuracy", "Precision", "Recall", "F1",
    "ROC_AUC", "PR_AUC", "MCC", "Brier"
]

cv_summary = pd.DataFrame({
    "Metric": metric_columns,
    "Mean": [fold_results_df[m].mean() for m in metric_columns],
    "SD": [fold_results_df[m].std(ddof=1) for m in metric_columns],
    "Variance": [fold_results_df[m].var(ddof=1) for m in metric_columns],
    "Min": [fold_results_df[m].min() for m in metric_columns],
    "Max": [fold_results_df[m].max() for m in metric_columns]
})

display(cv_summary)
cv_summary.to_csv(OUTPUT_DIR / "mol2vec_svm_5fold_summary.csv", index=False)

oof_df = train_df[["compound_id", "smiles", "label"]].copy()
oof_df["fold"] = oof_fold
oof_df["y_prob"] = oof_prob
oof_df["y_pred_0.5"] = (oof_prob >= 0.5).astype(int)
oof_df.to_csv(OUTPUT_DIR / "mol2vec_svm_oof_predictions.csv", index=False)


## 6. Cross-validation ROC and precision–recall curves

In [ ]:

plt.figure(figsize=(7, 6))
for fold in range(1, CONFIG["cv_folds"] + 1):
    mask = oof_fold == fold
    fpr, tpr, _ = roc_curve(train_df.loc[mask, "label"], oof_prob[mask])
    auc_value = roc_auc_score(train_df.loc[mask, "label"], oof_prob[mask])
    plt.plot(fpr, tpr, label=f"Fold {fold} (AUC={auc_value:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Mol2Vec+SVM: 5-Fold ROC Curves")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "mol2vec_svm_5fold_roc.png", dpi=300)
plt.show()


In [ ]:

plt.figure(figsize=(7, 6))
for fold in range(1, CONFIG["cv_folds"] + 1):
    mask = oof_fold == fold
    precision, recall, _ = precision_recall_curve(
        train_df.loc[mask, "label"], oof_prob[mask]
    )
    ap_value = average_precision_score(
        train_df.loc[mask, "label"], oof_prob[mask]
    )
    plt.plot(recall, precision, label=f"Fold {fold} (AP={ap_value:.3f})")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Mol2Vec+SVM: 5-Fold Precision–Recall Curves")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "mol2vec_svm_5fold_pr.png", dpi=300)
plt.show()


## 7. Final model, validation threshold selection, and independent test evaluation

In [ ]:

# Learn the final Mol2Vec representation using only the 72% training subset.
final_m2v = train_mol2vec(
    train_df["smiles"],
    config=CONFIG["mol2vec"],
    seed=SEED
)

X_train = featurize_smiles(
    train_df["smiles"], final_m2v, radius=CONFIG["mol2vec"]["radius"]
)
X_validation = featurize_smiles(
    validation_df["smiles"], final_m2v, radius=CONFIG["mol2vec"]["radius"]
)
X_test = featurize_smiles(
    test_df["smiles"], final_m2v, radius=CONFIG["mol2vec"]["radius"]
)

final_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(**CONFIG["svm"]))
])
final_pipeline.fit(X_train, train_df["label"])

validation_prob = final_pipeline.predict_proba(X_validation)[:, 1]
test_prob = final_pipeline.predict_proba(X_test)[:, 1]

print("Final Mol2Vec vocabulary size:", len(final_m2v.wv))
print("Embedding dimension:", final_m2v.vector_size)


In [ ]:

def select_f1_threshold(y_true, y_prob):
    thresholds = np.linspace(0.01, 0.99, 99)
    scores = [
        f1_score(y_true, (y_prob >= threshold).astype(int), zero_division=0)
        for threshold in thresholds
    ]
    best_index = int(np.argmax(scores))
    return float(thresholds[best_index]), float(scores[best_index])


best_threshold, best_validation_f1 = select_f1_threshold(
    validation_df["label"].to_numpy(),
    validation_prob
)

validation_metrics = calculate_metrics(
    validation_df["label"].to_numpy(),
    validation_prob,
    threshold=best_threshold
)
test_metrics = calculate_metrics(
    test_df["label"].to_numpy(),
    test_prob,
    threshold=best_threshold
)

final_metrics_df = pd.DataFrame([
    {"Subset": "Validation", "Threshold": best_threshold, **validation_metrics},
    {"Subset": "Independent test", "Threshold": best_threshold, **test_metrics}
])

print("Validation-selected threshold:", best_threshold)
display(final_metrics_df)
final_metrics_df.to_csv(
    OUTPUT_DIR / "mol2vec_svm_validation_test_metrics.csv",
    index=False
)


## 8. Test predictions, ADME descriptors, and docking-preparation candidates

In [ ]:

def calculate_adme(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {
            "MW": np.nan, "LogP": np.nan, "HBD": np.nan, "HBA": np.nan,
            "TPSA": np.nan, "RotBonds": np.nan, "QED": np.nan,
            "Lipinski_pass": False
        }

    mw = Descriptors.MolWt(mol)
    logp = Crippen.MolLogP(mol)
    hbd = Lipinski.NumHDonors(mol)
    hba = Lipinski.NumHAcceptors(mol)
    tpsa = rdMolDescriptors.CalcTPSA(mol)
    rot = Lipinski.NumRotatableBonds(mol)
    qed = QED.qed(mol)
    lipinski_pass = (mw <= 500) and (logp <= 5) and (hbd <= 5) and (hba <= 10)

    return {
        "MW": mw,
        "LogP": logp,
        "HBD": hbd,
        "HBA": hba,
        "TPSA": tpsa,
        "RotBonds": rot,
        "QED": qed,
        "Lipinski_pass": lipinski_pass
    }


test_predictions = test_df[["compound_id", "smiles", "label"]].copy()
test_predictions["y_prob"] = test_prob
test_predictions["y_pred"] = (test_prob >= best_threshold).astype(int)

adme_df = pd.DataFrame(
    [calculate_adme(s) for s in test_predictions["smiles"]]
)
test_predictions = pd.concat(
    [test_predictions.reset_index(drop=True), adme_df],
    axis=1
)
test_predictions = test_predictions.sort_values(
    ["y_prob", "QED"], ascending=[False, False]
).reset_index(drop=True)

test_predictions.to_csv(
    OUTPUT_DIR / "mol2vec_svm_test_predictions_with_adme.csv",
    index=False
)

top_candidates = (
    test_predictions
    .query("y_pred == 1")
    .head(20)
    .copy()
)
top_candidates.to_csv(
    OUTPUT_DIR / "mol2vec_svm_top20_docking_candidates.csv",
    index=False
)

display(top_candidates.head(10))


## 9. Save models and reproducibility metadata

In [ ]:

final_m2v.save(str(OUTPUT_DIR / "mol2vec_word2vec_300d.model"))
dump(final_pipeline, OUTPUT_DIR / "mol2vec_svm_pipeline.joblib")

with open(OUTPUT_DIR / "configuration.json", "w") as file:
    json.dump(CONFIG, file, indent=2)

table2_row = {
    "Model": "Mol2Vec+SVM",
    "Loss": "hinge (SVC)",
    "Optimizer": "---",
    "LR": "---",
    "WD": "---",
    "Dropout": "---",
    "Epochs": "---",
    "Batch": "---",
    "Representation / architecture": (
        "Mol2Vec: Morgan identifiers r=1; skip-gram Word2Vec, "
        "300 dimensions, window=10, min_count=1, negative=5, 30 epochs; "
        "molecular vector=sum of token vectors; StandardScaler; "
        "SVC(RBF, C=1.0, gamma=scale, class_weight=balanced)"
    )
}
pd.DataFrame([table2_row]).to_csv(
    OUTPUT_DIR / "table2_mol2vec_svm_row.csv",
    index=False
)
display(pd.DataFrame([table2_row]))



## 10. Recommended Table 2 LaTeX row

After the notebook has been run without modifications, the following row accurately represents the implemented configuration:

```latex
\midrule
Mol2Vec+SVM &
hinge (SVC) & --- & --- & --- & --- & --- & --- &
Mol2Vec Morgan identifiers ($r{=}1$); skip-gram Word2Vec
(300 dimensions, window${=}10$, \texttt{min\_count}${=}1$,
negative${=}5$, 30 epochs); sum pooling; \texttt{StandardScaler};
\texttt{SVC}(\texttt{RBF}, $C{=}1.0$, $\gamma{=}$\texttt{scale},
\texttt{class\_weight}${=}$\texttt{balanced}) \\
```

Because the neural embedding is trained separately from the SVC classifier, the table’s `Epochs` column is left as `---` for the classifier. The 30 Word2Vec training epochs are stated explicitly in the representation/architecture column.


## 11. Archive all outputs

In [ ]:

import shutil

archive_path = shutil.make_archive(
    "Mol2Vec_SVM_HIV7_5Fold_Outputs",
    "zip",
    OUTPUT_DIR
)
print("Created:", archive_path)
